<a href="https://colab.research.google.com/github/annatsamoyra-prog/data-story-/blob/main/cnn_gr_tech_2026_scraper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q requests beautifulsoup4 pandas numpy


In [ ]:
# για το scraping
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

# για τα nan values
import numpy as np


In [ ]:
main_url = "https://www.cnn.gr"
category_url = "https://www.cnn.gr/tech"

start_page = 1
end_page = 8

TARGET_YEAR = 2026

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    ),
    "Accept-Language": "el-GR,el;q=0.9,en-US;q=0.8,en;q=0.7",
}

# Ένα άρθρο tech είναι πάντα /tech/story|audio|longform/<αριθμός>/<slug>
ARTICLE_PATH_RE = re.compile(r"^/tech/(story|audio|longform)/\d+/")

# Boilerplate ενότητες -- ολόκληρη η παράγραφος/heading πρέπει να ταιριάζει, μόλις τη συναντήσουμε
# σταματάμε να μαζεύουμε κείμενο (επιβεβαιωμένο από πραγματικό άρθρο, όχι εικασία).
STOP_HEADINGS = ["ΣΧΕΤΙΚΕΣ ΕΙΔΗΣΕΙΣ", "ΡΟΗ ΕΙΔΗΣΕΩΝ", "ΔΗΜΟΦΙΛΗ"]

# Γραμμές που παραλείπουμε αλλά ΔΕΝ σταματάμε (μπορεί να εμφανιστούν νωρίς μέσα στο άρθρο)
JUNK_LINE_PREFIXES = ["Δες περισσότερα άρθρα του CNN Greece", "Ακολουθήστε το CNN Greece"]


## ΒΗΜΑ 3: Μαζεύουμε τα urls των άρθρων (teasers)

Λούπα στις σελίδες 1-8, `find_all` στο container `div.list-items.tall-cards`, `story_dict` με
μόνο το `url`.


In [ ]:
teasers_list = []

for page_num in range(start_page, end_page + 1):
    page_url = category_url if page_num == 1 else f"{category_url}?page={page_num}"
    print(f"Σελίδα {page_num}: {page_url}")

    response = requests.get(page_url, headers=HEADERS)
    doc = BeautifulSoup(response.text, "html.parser")

    #*** teaser_container -- εδώ είναι το container των teasers στο cnn.gr/tech ***
    teaser_container = doc.find("div", {"class": "list-items"})
    links = teaser_container.find_all("a", href=True) if teaser_container else []

    for link_tag in links:
        href = link_tag["href"]
        if not ARTICLE_PATH_RE.match(href):
            continue  # όχι άρθρο tech (π.χ. tag link, section link κ.λπ.)
        story_dict = {"url": main_url + href}
        teasers_list.append(story_dict)

    time.sleep(1)

cnn_teasers_df = pd.DataFrame(teasers_list)
cnn_teasers_df = cnn_teasers_df.drop_duplicates(subset=["url"]).reset_index(drop=True)
print("\nΣυνολικά teasers:", len(cnn_teasers_df))
cnn_teasers_df


Σελίδα 1: https://www.cnn.gr/tech
Σελίδα 2: https://www.cnn.gr/tech?page=2
Σελίδα 3: https://www.cnn.gr/tech?page=3
Σελίδα 4: https://www.cnn.gr/tech?page=4
Σελίδα 5: https://www.cnn.gr/tech?page=5
Σελίδα 6: https://www.cnn.gr/tech?page=6
Σελίδα 7: https://www.cnn.gr/tech?page=7
Σελίδα 8: https://www.cnn.gr/tech?page=8

Συνολικά teasers: 236


,url
0,https://www.cnn.gr/tech/story/553061/sta-4-8-d...
1,https://www.cnn.gr/tech/story/552497/oi-anthro...
2,https://www.cnn.gr/tech/story/552699/kai-i-kin...
3,https://www.cnn.gr/tech/story/552673/poso-efik...
4,https://www.cnn.gr/tech/story/552635/xiaomi-al...
...,...
231,https://www.cnn.gr/tech/story/508452/openai-se...
232,https://www.cnn.gr/tech/story/508557/poleis-xo...
233,https://www.cnn.gr/tech/story/508583/symfonia-...
234,https://www.cnn.gr/tech/story/508403/google-se...


## ΒΗΜΑ 4: Δοκιμή σε ένα άρθρο



In [ ]:
article_url = cnn_teasers_df.loc[0, "url"]
print(article_url)

response = requests.get(article_url, headers=HEADERS)
doc = BeautifulSoup(response.text, "html.parser")

# Το html tag <article> είναι ο container του κυρίως άρθρου (επιβεβαιωμένο από το CSS του site:
# article.main{padding:...}). Αν δεν βρεθεί, πέφτουμε πίσω σε όλη τη σελίδα.
article = doc.find("article")
if article is None:
    article = doc

print(article.prettify()[:3000])


https://www.cnn.gr/tech/story/553061/sta-4-8-dis-oi-xristes-tou-mobile-internet-dievrynetai-ostoso-to-psifiako-xasma
<article class="main news-story" id="news-item-553061" role="main">
 <header class="main-item-header">
  <a class="main-item-category" href="/tech">
   TECH
  </a>
  <h1 class="main-title">
   Στα 4,8 δισ. οι χρήστες του mobile internet - Διευρύνεται ωστόσο το ψηφιακό χάσμα
  </h1>
  <div class="main-details">
   <p class="main-author story-author">
    <a class="author-avatar" href="/profiles/orestis-panteloglou" title="Ορέστης Παντελόγλου">
     <img alt="Ορέστης Παντελόγλου" src="https://img.cnngreece.gr/img/120/120/90/2025/01/29/34098663-avatar120x120_Orestis_Panteloglou.jpg?t=yXdtPhwO7n3mo5NVGgCLMA"/>
    </a>
    <span class="is-curator">
     Επιμέλεια -
    </span>
    <a class="author-name" href="/profiles/orestis-panteloglou" title="Ορέστης Παντελόγλου">
     Ορέστης Παντελόγλου
    </a>
    - CNN Greece
   </p>
   <p class="dateline story-dateline">
    <svg h

In [ ]:
# Τίτλος -- προτιμάμε το og:title meta (πάντα καθαρό), αλλιώς το h1
og_title = doc.find("meta", {"property": "og:title"})
title = og_title["content"] if og_title else (article.find("h1").text if article.find("h1") else None)
title


'Στα 4,8 δισ. οι χρήστες του mobile internet - Διευρύνεται ωστόσο το ψηφιακό χάσμα'

In [ ]:
# Ημερομηνία -- meta property="article:published_time", ISO format με timezone, έτοιμο για
# pd.to_datetime χωρίς επεξεργασία.
meta_date = doc.find("meta", {"property": "article:published_time"})
date = meta_date["content"] if meta_date else None
date


'2026-09-19T14:37:09+03:00'

In [ ]:
# Συντάκτης -- meta name="author"
meta_author = doc.find("meta", {"name": "author"})
author = meta_author["content"] if meta_author else None
author


'Ορέστης Παντελόγλου'

In [ ]:
# Πλήρες κείμενο -- μαζεύουμε p/h2 μέχρι να συναντήσουμε boilerplate ενότητα.
p_texts_list = []
for el in article.find_all(["p", "h2", "h3"]):
    text = el.get_text(" ", strip=True)
    if not text:
        continue
    if text in STOP_HEADINGS or any(text.startswith(h) for h in STOP_HEADINGS):
        break
    if any(text.startswith(prefix) for prefix in JUNK_LINE_PREFIXES):
        continue
    p_texts_list.append(text)

full_text = " ".join(p_texts_list)
full_text = "".join(full_text.splitlines())
full_text


'Επιμέλεια - Ορέστης Παντελόγλου - CNN Greece Σάββατο, 19 Σεπτεμβρίου 2026 14:37 Στα 4,8 δισεκατομμύρια έφθασε το 2025 ο αριθμός των ανθρώπων που χρησιμοποιούν το mobile internet (πρόσβαση στο διαδίκτυο μέσω κινητού τηλεφώνου) μέσω δικής τους συσκευής, αντιπροσωπεύοντας πλέον το 59% του παγκόσμιου πληθυσμού. Ωστόσο, παρά τη διεύρυνση της συνδεσιμότητας και την ταχεία ανάπτυξη των δικτύων 5G, περίπου 3,4 δισεκατομμύρια άνθρωποι εξακολουθούν να μη χρησιμοποιούν το διαδίκτυο μέσω κινητού τηλεφώνου. Τα στοιχεία περιλαμβάνονται στην έκθεση «The State of Mobile Internet Connectivity 2026» της GSMA, που δόθηκε στη δημοσιότητα, η οποία αποτυπώνει την πρόοδο που έχει σημειωθεί, αλλά και τις μεγάλες οικονομικές, κοινωνικές και γεωγραφικές ανισότητες που παραμένουν. Το mobile internet αντιστοιχεί στο 84% των ευρυζωνικών συνδέσεων παγκοσμίως και, ιδιαίτερα στις χώρες χαμηλού και μεσαίου εισοδήματος, αποτελεί συχνά το βασικό ή ακόμη και το μοναδικό μέσο πρόσβασης στο διαδίκτυο. Η αύξηση των χρηστών

## ΒΗΜΑ 5: Η λούπα για όλα τα άρθρα

In [ ]:
full_articles_list = []

for article_url in cnn_teasers_df["url"]:
    response = requests.get(article_url, headers=HEADERS)
    doc = BeautifulSoup(response.text, "html.parser")

    article = doc.find("article")
    if article is None:
        article = doc

    full_article_dict = {}
    full_article_dict["site"] = "cnn.gr"
    full_article_dict["url"] = article_url

    try:
        og_title = doc.find("meta", {"property": "og:title"})
        title = og_title["content"] if og_title else article.find("h1").text
        full_article_dict["title"] = title
    except Exception:
        full_article_dict["title"] = None

    try:
        meta_date = doc.find("meta", {"property": "article:published_time"})
        full_article_dict["date"] = meta_date["content"] if meta_date else None
    except Exception:
        full_article_dict["date"] = None

    try:
        meta_author = doc.find("meta", {"name": "author"})
        full_article_dict["author"] = meta_author["content"] if meta_author else None
    except Exception:
        full_article_dict["author"] = None

    try:
        p_texts_list = []
        for el in article.find_all(["p", "h2", "h3"]):
            text = el.get_text(" ", strip=True)
            if not text:
                continue
            if text in STOP_HEADINGS or any(text.startswith(h) for h in STOP_HEADINGS):
                break
            if any(text.startswith(prefix) for prefix in JUNK_LINE_PREFIXES):
                continue
            p_texts_list.append(text)
        full_text = " ".join(p_texts_list)
        full_text = "".join(full_text.splitlines())
        full_article_dict["full_text"] = full_text if full_text else None
    except Exception:
        full_article_dict["full_text"] = None

    full_articles_list.append(full_article_dict)
    time.sleep(1)

cnn_full_articles_df = pd.DataFrame(full_articles_list)
cnn_full_articles_df = cnn_full_articles_df[["site", "url", "title", "date", "author", "full_text"]]
cnn_full_articles_df


,site,url,title,date,author,full_text
0,cnn.gr,https://www.cnn.gr/tech/story/553061/sta-4-8-d...,"Στα 4,8 δισ. οι χρήστες του mobile internet - ...",2026-09-19T14:37:09+03:00,Ορέστης Παντελόγλου,Επιμέλεια - Ορέστης Παντελόγλου - CNN Greece Σ...
1,cnn.gr,https://www.cnn.gr/tech/story/552497/oi-anthro...,Οι άνθρωποι αναπτύσσουν αληθινά αισθήματα για ...,2026-09-19T08:00:30+03:00,Newsroom,"Newsroom Σάββατο, 19 Σεπτεμβρίου 2026 08:00 Γι..."
2,cnn.gr,https://www.cnn.gr/tech/story/552699/kai-i-kin...,Και η Κίνα πιστεύει ότι η ΑΙ θα μπορούσε να μα...,2026-09-18T08:02:00+03:00,Newsroom,"Newsroom Παρασκευή, 18 Σεπτεμβρίου 2026 08:02 ..."
3,cnn.gr,https://www.cnn.gr/tech/story/552673/poso-efik...,Πόσο εφικτό είναι να επιβραδύνουμε την ανάπτυξ...,2026-09-17T10:12:35+03:00,Newsroom,"Newsroom Πέμπτη, 17 Σεπτεμβρίου 2026 10:12 Η ι..."
4,cnn.gr,https://www.cnn.gr/tech/story/552635/xiaomi-al...,Xiaomi: Αλλάζει ταχύτητα και ποντάρει στην αντ...,2026-09-17T07:46:57+03:00,Δημήτρης Μαλλάς,"Δημήτρης Μαλλάς - CNN Greece Πέμπτη, 17 Σεπτεμ..."
...,...,...,...,...,...,...
231,cnn.gr,https://www.cnn.gr/tech/story/508452/openai-se...,OpenAI: Σε «κόκκινο συναγερμό» το ChatGPT λόγω...,2025-12-05T14:05:05+02:00,Νατάσα Βορύλλα,Επιμέλεια - Νατάσα Βορύλλα - CNN Greece Παρασκ...
232,cnn.gr,https://www.cnn.gr/tech/story/508557/poleis-xo...,Πόλεις χωρίς εμπόδια: Πώς η τεχνολογία ενισχύε...,2025-12-05T11:00:47+02:00,Newsroom,"Newsroom Παρασκευή, 05 Δεκεμβρίου 2025 11:00 Η..."
233,cnn.gr,https://www.cnn.gr/tech/story/508583/symfonia-...,"Συμφωνία 3,9 δισ. ευρώ για κέντρο τεχνητής νοη...",2025-12-05T10:44:32+02:00,Νατάσα Βορύλλα,Επιμέλεια - Νατάσα Βορύλλα - CNN Greece Παρασκ...
234,cnn.gr,https://www.cnn.gr/tech/story/508403/google-se...,Google: «Σεισμός τώρα» η πιο δημοφιλής αναζήτη...,2025-12-04T10:40:01+02:00,Νατάσα Βορύλλα,Επιμέλεια - Νατάσα Βορύλλα - CNN Greece Πέμπτη...


In [ ]:
len(cnn_full_articles_df )

236

## ΒΗΜΑ 6: Καθαρισμός -- κενές γραμμές & διπλότυπα

In [ ]:
nan_rows = cnn_full_articles_df[cnn_full_articles_df["full_text"].isna()]
print("Κενές γραμμές full_text:", len(nan_rows))

cnn_full_articles_df = cnn_full_articles_df.dropna(subset=["full_text"]).reset_index(drop=True)


Κενές γραμμές full_text: 0


In [ ]:
duplicate_rows = cnn_full_articles_df[cnn_full_articles_df.duplicated(subset=["full_text"], keep=False)]
print("Διπλές εγγραφές:", len(duplicate_rows))

cnn_full_articles_df = cnn_full_articles_df.drop_duplicates(subset="full_text", keep="first").reset_index(drop=True)


Διπλές εγγραφές: 0


## ΒΗΜΑ 7: datetime + τελικό φίλτρο 2026

In [ ]:
cnn_full_articles_df["datetime"] = pd.to_datetime(cnn_full_articles_df["date"], errors="coerce", utc=True)

# Ρητό φίλτρο έτους -- γραμμές με άκυρη/άγνωστη ημερομηνία (NaT) αποκλείονται αυτόματα εδώ.
cnn_2026_df = cnn_full_articles_df[cnn_full_articles_df["datetime"].dt.year == TARGET_YEAR].copy()
cnn_2026_df = cnn_2026_df.sort_values("datetime").reset_index(drop=True)
cnn_2026_df = cnn_2026_df[["site", "url", "title", "date", "author", "full_text", "datetime"]]

print("Πλήθος άρθρων 2026:", len(cnn_2026_df))
cnn_2026_df.head()

Πλήθος άρθρων 2026: 214


,site,url,title,date,author,full_text,datetime
0,cnn.gr,https://www.cnn.gr/tech/story/512652/ti-einai-...,Τι είναι το «AI Slop»: Αποτελεί πάνω από το 20...,2026-01-01T23:57:00+02:00,Καλλιρόη Πεπονή,Επιμέλεια - Καλλιρόη Πεπονή - CNN Greece Πέμπτ...,2026-01-01 21:57:00+00:00
1,cnn.gr,https://www.cnn.gr/tech/story/513004/to-doro-t...,Το... δώρο του Ίλον Μασκ στη Βενεζουέλα - Δωρε...,2026-01-04T23:19:00+02:00,Newsroom,"Newsroom Κυριακή, 04 Ιανουαρίου 2026 23:19 Στο...",2026-01-04 21:19:00+00:00
2,cnn.gr,https://www.cnn.gr/tech/story/513029/giati-ako...,Γιατί ακόμη και η ίδια η ΑΙ δυσκολεύεται να ελ...,2026-01-05T10:22:34+02:00,Newsroom,"Newsroom Δευτέρα, 05 Ιανουαρίου 2026 10:22 Καθ...",2026-01-05 08:22:34+00:00
3,cnn.gr,https://www.cnn.gr/tech/story/513498/ta-sxedia...,Τα σχέδια της Apple για το… iPhone 21 αποκάλυψ...,2026-01-08T09:38:08+02:00,Newsroom,"Newsroom Πέμπτη, 08 Ιανουαρίου 2026 09:38 Η σε...",2026-01-08 07:38:08+00:00
4,cnn.gr,https://www.cnn.gr/tech/story/513505/to-rompot...,Το ρομπότ WALL-E της Pixar έγινε πραγματικότητ...,2026-01-08T10:55:15+02:00,Newsroom,"Newsroom Πέμπτη, 08 Ιανουαρίου 2026 10:55 Το ρ...",2026-01-08 08:55:15+00:00


## ΒΗΜΑ 8: Γρήγορος έλεγχος

In [ ]:
print("Σύνολο άρθρων:", len(cnn_2026_df))
print("\nMissing values ανά στήλη:")
print(cnn_2026_df.isna().sum())
print("\nΆρθρα ανά μήνα:")
print(cnn_2026_df["datetime"].dt.to_period("M").value_counts().sort_index())
print("\nMin datetime:", cnn_2026_df["datetime"].min())
print("Max datetime:", cnn_2026_df["datetime"].max())


Σύνολο άρθρων: 214

Missing values ανά στήλη:
site         0
url          0
title        0
date         0
author       0
full_text    0
datetime     0
dtype: int64

Άρθρα ανά μήνα:
datetime
2026-01    22
2026-02    16
2026-03    37
2026-04    20
2026-05    25
2026-06    15
2026-07    29
2026-08    25
2026-09    25
Freq: M, Name: count, dtype: int64

Min datetime: 2026-01-01 21:57:00+00:00
Max datetime: 2026-09-19 11:37:09+00:00


/tmp/ipykernel_1343/4072381664.py:5: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  print(cnn_2026_df["datetime"].dt.to_period("M").value_counts().sort_index())


In [ ]:
import base64
from google.colab import userdata

def save_df_to_github(df, repo, path, token=None, branch="main", message="Update dataset"):
    # If no token is provided, try to get it from Colab secrets, defaulting to GITHUB_TOKEN
    token = token or userdata.get("GITHUB_TOKEN")

    # If after this, token is still None, it means no secret was found or accessible.
    # In a non-interactive environment, userdata.get() often times out.
    if token is None:
        raise ValueError("GitHub token not found. Please set 'GITHUB_TOKEN' in Colab secrets or provide it manually.")

    csv_content = df.to_csv(index=False, encoding="utf-8-sig")
    content_b64 = base64.b64encode(csv_content.encode("utf-8-sig")).decode("utf-8")

    url = f"https://api.github.com/repos/{repo}/contents/{path}"
    headers = {"Authorization": f"token {token}", "Accept": "application/vnd.github+json"}

    existing = requests.get(url, headers=headers, params={"ref": branch})
    sha = existing.json().get("sha") if existing.status_code == 200 else None

    payload = {"message": message, "content": content_b64, "branch": branch}
    if sha:
        payload["sha"] = sha

    response = requests.put(url, headers=headers, json=payload)
    response.raise_for_status()
    print(f"✅ Αποθηκεύτηκε: https://github.com/{repo}/blob/{branch}/{path}")
    return response.json()

save_df_to_github(
    cnn_2026_df,
    repo="annatsamoyra-prog/data-story-",
    path="cnn_gr_tech_2026.csv",
    token=userdata.get("newtoken"),

✅ Αποθηκεύτηκε: https://github.com/annatsamoyra-prog/data-story-/blob/main/cnn_gr_tech_2026.csv


{'content': {'name': 'cnn_gr_tech_2026.csv',
  'path': 'cnn_gr_tech_2026.csv',
  'sha': '134a562921460db58c8e02c4bb0e20a1b59e1a14',
  'size': 1589395,
  'url': 'https://api.github.com/repos/annatsamoyra-prog/data-story-/contents/cnn_gr_tech_2026.csv?ref=main',
  'html_url': 'https://github.com/annatsamoyra-prog/data-story-/blob/main/cnn_gr_tech_2026.csv',
  'git_url': 'https://api.github.com/repos/annatsamoyra-prog/data-story-/git/blobs/134a562921460db58c8e02c4bb0e20a1b59e1a14',
  'download_url': 'https://raw.githubusercontent.com/annatsamoyra-prog/data-story-/main/cnn_gr_tech_2026.csv',
  'type': 'file',
  '_links': {'self': 'https://api.github.com/repos/annatsamoyra-prog/data-story-/contents/cnn_gr_tech_2026.csv?ref=main',
   'git': 'https://api.github.com/repos/annatsamoyra-prog/data-story-/git/blobs/134a562921460db58c8e02c4bb0e20a1b59e1a14',
   'html': 'https://github.com/annatsamoyra-prog/data-story-/blob/main/cnn_gr_tech_2026.csv'}},
 'commit': {'sha': 'bc2c75ba63712d58fe0a070f33